## CS3249 Tutorial – LangGraph Exercise Version

### Tasks Overview

| Task | Description | Concept Learned | Example Input | Expected Behavior |
|------|--------------|----------------|----------------|-------------------|
| **1. Fallback Logic** | Make the LangGraph automatically switch modes if one path fails. For example, if Graph RAG cannot find an answer, the system should automatically try Vector RAG or fall back to No-RAG. | *Control flow & reliability* | **Input:** “Who are Elon Musk’s coauthors?”  <br>→ Graph RAG fails (no data). | ✅ The system detects failure and automatically falls back to **No-RAG**, which generates a general answer such as “I don’t have data on that, but Elon Musk is known for leading companies like Tesla and SpaceX.” |
| **2. Caching Layer** | Add a cache so repeated queries are answered instantly without new LLM calls. | *Efficiency & cost-saving* | **Input:** Ask “Who are Yi-Chieh Lee’s coauthors?” twice in a row. | ✅ The first call queries Graph RAG normally. <br>✅ The second call returns the same answer instantly with a “cache hit” message in the logs. |
| **3. LLM-based Router** | Replace the keyword-based routing with an LLM agent that decides which reasoning mode to use. The router should interpret the question semantically, not just by string matching. | *Meta-reasoning & orchestration* | **Input:** “Show me Yi-Chieh Lee’s collaboration network.” | ✅ The router LLM interprets “collaboration network” as a **Graph RAG** question and routes it correctly — even though the word “coauthor” isn’t explicitly mentioned. |

Each section of the code includes clear markers like `#TODO`, where you’ll fill in your implementation.

### 1. Imports & Environment Setup

In [ ]:
import os
import datetime
import time
from dotenv import load_dotenv
from fastapi import FastAPI
from pydantic import BaseModel
import nest_asyncio
import uvicorn

# LangChain & OpenAI
from langchain_openai import ChatOpenAI

 ### 2. Load Environment Variables

In [ ]:
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

 ### 3. Initialize LLM

In [ ]:
llm = ChatOpenAI(
    model=OPENAI_MODEL,
    openai_api_key=OPENAI_API_KEY,
    openai_api_base=OPENAI_BASE_URL,
    temperature=0,
)

 ### 4.1 Vector RAG Setup

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain.text_splitter import CharacterTextSplitter

loader = TextLoader("../data/vector_sample_data.txt")
docs = loader.load()
splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=50)
split_docs = splitter.split_documents(docs)

embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)
vectordb = Chroma.from_documents(split_docs, embeddings)
retriever = vectordb.as_retriever()

rag_chain = RetrievalQA.from_chain_type(
    llm=llm, retriever=retriever, return_source_documents=True
)

print("✅ Vector RAG ready")

 ### 4.2 Graph RAG Setup

In [ ]:
import networkx as nx
import json
from langchain.chains import GraphQAChain
from langchain_community.graphs import NetworkxEntityGraph

GRAPH_PATH = "../data/graph_sample_data.json"
with open(GRAPH_PATH, "r") as f:
    graph_data = json.load(f)

G = nx.node_link_graph(graph_data, directed=True)
if not isinstance(G, nx.DiGraph):
    G = nx.DiGraph(G)

graph = NetworkxEntityGraph(G)
graph_chain = GraphQAChain.from_llm(llm=llm, graph=graph, verbose=True)
print(f"✅ Graph RAG ready ({len(G.nodes())} nodes, {len(G.edges())} edges)")

 ### 4.3 LangGraph Router Agent (Exercise Section)

In [ ]:
from langgraph.graph import StateGraph, END
from langchain.schema import HumanMessage

# ====== Simple cache for all modes ======
cache = {}  #TODO (Task 2): Use this dict to store and reuse previous answers


# ---------- Core chain calls ----------
def call_no_rag(state):
    query = state["query"]
    #TODO (Task 2): Check cache before invoking the model
    response = llm.invoke(query)
    result = {"answer": response.content, "mode": "No-RAG"}
    #TODO (Task 2): Save result to cache
    return result


def call_rag(state):
    query = state["query"]
    response = rag_chain.invoke(query)
    answer = response["result"]

    #TODO (Task 2): If no relevant docs or generic response, mark fallback
    fallback = "I don't know" in answer or "No answer" in answer
    return {"answer": answer, "mode": "Vector RAG", "fallback": fallback}


def call_graph_rag(state):
    query = state["query"]
    response = graph_chain.invoke({"query": query})
    answer = response.get("result", "No answer found.")
    fallback = "No answer" in answer
    return {"answer": answer, "mode": "Graph RAG", "fallback": fallback}


# ---------- Router ----------
def router_node(state):
    return state


#TODO (Task 3): Implement an LLM-based router
# Hint:
#  - Import PromptTemplate and LLMChain
#  - Create a router prompt asking which mode to use: [no_rag, rag, graph_rag]
#  - Call router_chain.invoke({"query": state["query"]})
#  - Return the chosen mode string (lowercase)

# Placeholder keyword-based router (for now)
def route_query(state):
    q = state["query"].lower()
    if "coauthor" in q or "network" in q:
        return "graph_rag"
    elif "assignment" in q or "deadline" in q or "cs3249" in q:
        return "rag"
    else:
        return "no_rag"


# ---------- Build LangGraph ----------
workflow = StateGraph(dict)

workflow.add_node("start", router_node)
workflow.add_node("no_rag", call_no_rag)
workflow.add_node("rag", call_rag)
workflow.add_node("graph_rag", call_graph_rag)

# Base routing
workflow.add_conditional_edges("start", route_query, {
    "no_rag": "no_rag",
    "rag": "rag",
    "graph_rag": "graph_rag"
})

#TODO (Task 1): Add fallback edges
# Example:
# workflow.add_conditional_edges("graph_rag", lambda s: "rag" if s.get("fallback") else END)
# workflow.add_conditional_edges("rag", lambda s: "no_rag" if s.get("fallback") else END)

for node in ["no_rag", "rag", "graph_rag"]:
    workflow.add_edge(node, END)

workflow.set_entry_point("start")
langgraph_app = workflow.compile()
print("✅ LangGraph router ready (exercise version)")

 ### 5. FastAPI App Setup

In [ ]:
app = FastAPI()

class Query(BaseModel):
    message: str

# ------------------------------------------------------------
# (A) No-RAG Endpoint: Direct LLM response without retrieval
# ------------------------------------------------------------
@app.post("/chat/no_rag")
def api_no_rag(query: Query):
    response = llm.invoke(query.message)
    return {"answer": response.content}


# ------------------------------------------------------------
# (B) Vector RAG Endpoint: Use retriever + context-augmented generation
# ------------------------------------------------------------
@app.post("/chat/rag")
def api_rag(query: Query):
    response = rag_chain.invoke(query.message)
    answer = response["result"]
    sources = [doc.metadata.get("source", "sample.txt") for doc in response["source_documents"]]
    return {"answer": answer, "sources": list(set(sources))}


# ------------------------------------------------------------
# (C) Example Tool Endpoint
# ------------------------------------------------------------
@app.get("/tools")
def list_tools():
    return {
        "tools": [
            {
                "name": "get_current_time",
                "description": "Return the current server time",
                "input_schema": {"type": "object", "properties": {}, "required": []},
            }
        ]
    }

@app.post("/call/get_current_time")
def get_current_time():
    now = datetime.datetime.now().isoformat()
    return {"answer": now}



# ------------------------------------------------------------
# (D) Graph RAG Endpoint
# ------------------------------------------------------------

@app.post("/chat/graph_rag")
def api_graph_rag(query: Query):
    print(f"Received query: {query.message}")
    response = graph_chain.invoke({"query": query.message})
    answer = response.get("result", "No answer found.")
    return {"answer": answer}


# ------------------------------------------------------------
# (E) Router Agent
# ------------------------------------------------------------

@app.post("/chat/langgraph")
def api_langgraph(query: Query):
    print(f"🧭 LangGraph routing query: {query.message}")
    result = langgraph_app.invoke({"query": query.message})
    return {"answer": result.get("answer"), "mode": result.get("mode", "Unknown")}

 ### 6. Run Server (for Jupyter)

In [ ]:
nest_asyncio.apply()
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()